### Section D

###D1

In [0]:
dbutils.widgets.text("catalog", "fk_ce_2026_07_1378")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql import functions as F

df_stores = spark.read.table(f"{catalog}.bronze.stores")

df_clean = df_stores \
    .withColumn("store_id", F.regexp_replace(F.col("store_id"), r"(?<=[A-Za-z])0(?=\d)", "")) \
    .withColumn("store_name", F.trim(F.col("store_name"))) \
    .withColumn("city", F.initcap(F.trim(F.col("city")))) \
    .withColumn("state", F.trim(F.col("state"))) \
    .withColumn("region", F.trim(F.col("region")))

df_clean.write.mode("overwrite").saveAsTable(f"{catalog}.silver.stores")


###D2

In [0]:
dbutils.widgets.text("current_snapshot_date", "2023-07-13")
current_snapshot_date = dbutils.widgets.get("current_snapshot_date")

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

df_bronze_source = spark.read.table(f"{catalog}.bronze.products")

df_updates = df_bronze_source.select(
    F.regexp_replace(F.col("product_id"), r"(?<=[A-Za-z])0(?=\d)", "").alias("product_id"),
    F.col("product_name"),
    F.col("brand"),
    F.col("category.l1").alias("category_l1"),
    F.col("category.l2").alias("category_l2"),
    F.col("pack.size").cast("int").alias("pack_size"),
    F.col("pack.unit").alias("pack_unit"),
    F.col("mrp").cast("decimal(10,2)"),
    F.col("selling_price").cast("decimal(10,2)"),
    F.col("is_active").cast("boolean"),
    F.col("tags"),
    F.lit(current_snapshot_date).cast("date").alias("last_changed_snapshot_date")
)

target_table_name = f"{catalog}.silver.products"

if not spark.catalog.tableExists(target_table_name) or len(spark.read.table(target_table_name).columns) == 0:
    df_updates.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table_name)

silver_table = DeltaTable.forName(spark, target_table_name)

(
    silver_table.alias("target")
    .merge(
        df_updates.alias("source"),
        "target.product_id = source.product_id"
    )
    .whenMatchedUpdate(
        condition=(
            "(target.product_name <=> source.product_name) = False OR "
            "(target.brand <=> source.brand) = False OR "
            "(target.category_l1 <=> source.category_l1) = False OR "
            "(target.category_l2 <=> source.category_l2) = False OR "
            "(target.pack_size <=> source.pack_size) = False OR "
            "(target.pack_unit <=> source.pack_unit) = False OR "
            "(target.mrp <=> source.mrp) = False OR "
            "(target.selling_price <=> source.selling_price) = False OR "
            "(target.is_active <=> source.is_active) = False OR "
            "(target.tags <=> source.tags) = False"
        ),
        set={
            "product_name": "source.product_name",
            "brand": "source.brand",
            "category_l1": "source.category_l1",
            "category_l2": "source.category_l2",
            "pack_size": "source.pack_size",
            "pack_unit": "source.pack_unit",
            "mrp": "source.mrp",
            "selling_price": "source.selling_price",
            "is_active": "source.is_active",
            "tags": "source.tags",
            "last_changed_snapshot_date": "source.last_changed_snapshot_date"
        }
    )
    .whenNotMatchedInsert(
        values={
            "product_id": "source.product_id",
            "product_name": "source.product_name",
            "brand": "source.brand",
            "category_l1": "source.category_l1",
            "category_l2": "source.category_l2",
            "pack_size": "source.pack_size",
            "pack_unit": "source.pack_unit",
            "mrp": "source.mrp",
            "selling_price": "source.selling_price",
            "is_active": "source.is_active",
            "tags": "source.tags",
            "last_changed_snapshot_date": "source.last_changed_snapshot_date"
        }
    )
    .execute()
)


In [0]:
%sql
ALTER TABLE IDENTIFIER(:catalog || '.silver.products') SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

###D3

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

df_bronze_customers = spark.read.table(f"{catalog}.bronze.customers")

df_cleaned = df_bronze_customers.select(
    F.regexp_replace(F.col("customer_id"), r"(?<=[A-Za-z])0(?=\d)", "").alias("customer_id"),
    F.trim(F.col("full_name")).alias("full_name"),
    F.lower(F.trim(F.col("email"))).alias("email"),
    F.substring(F.regexp_replace(F.col("phone"), r"\D", ""), -10, 10).alias("phone"),
    F.initcap(F.trim(F.col("city"))).alias("city"),
    F.upper(F.trim(F.col("tier"))).alias("tier"),
    F.coalesce(
        F.to_date(F.col("signup_date"), "yyyy-MM-dd"),
        F.to_date(F.col("signup_date"), "dd/MM/yyyy")
    ).alias("signup_date")
).distinct()


In [0]:
target_table_name = f"{catalog}.silver.customers"

if not spark.catalog.tableExists(target_table_name) or len(spark.read.table(target_table_name).columns) == 0:
    df_updates.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table_name)

silver_customers_table = DeltaTable.forName(spark, target_table_name)
df_target_current = silver_customers_table.toDF().filter("is_current = true")

df_evaluated_changes = df_cleaned.alias("src").join(
    df_target_current.alias("tgt"),
    "customer_id",
    "left"
).select(
    "src.*",
    F.expr(
        "(tgt.customer_id IS NOT NULL) AND "
        "((tgt.city <=> src.city) = False OR (tgt.tier <=> src.tier) = False)"
    ).alias("is_type2_change"),
    F.expr(
        "(tgt.customer_id IS NOT NULL) AND "
        "(tgt.city <=> src.city) AND (tgt.tier <=> src.tier) AND ("
        "(tgt.full_name <=> src.full_name) = False OR "
        "(tgt.email <=> src.email) = False OR "
        "(tgt.phone <=> src.phone) = False)"
    ).alias("is_type1_only_change")
)


df_pure_updates = df_evaluated_changes.filter("is_type2_change = false")


df_type2_closures = df_evaluated_changes.filter("is_type2_change = true").select(
    "customer_id", "full_name", "email", "phone", "city", "tier", "signup_date"
).withColumn("merge_key", F.col("customer_id")) 


df_type2_new_records = df_evaluated_changes.filter("is_type2_change = true").select(
    "customer_id", "full_name", "email", "phone", "city", "tier", "signup_date"
).withColumn("merge_key", F.lit(None).cast("string")) 

df_final_staging = df_pure_updates.withColumn("merge_key", F.col("customer_id")) \
    .unionByName(df_type2_closures, allowMissingColumns=True) \
    .unionByName(df_type2_new_records, allowMissingColumns=True)

(
    silver_customers_table.alias("target")
    .merge(
        df_final_staging.alias("source"),
        "target.customer_id = source.merge_key AND target.is_current = true"
    )
    .whenMatchedUpdate(
        condition="source.is_type2_change = true",
        set={
            "effective_to": F.lit(change_effective_date).cast("date"),
            "is_current": F.lit(False)
        }
    )
    .whenMatchedUpdate(
        condition="source.is_type1_only_change = true",
        set={
            "full_name": "source.full_name",
            "email": "source.email",
            "phone": "source.phone",
            "signup_date": "source.signup_date"
        }
    )
    .whenNotMatchedInsert(
        values={
            "customer_id": "source.customer_id",
            "full_name": "source.full_name",
            "email": "source.email",
            "phone": "source.phone",
            "city": "source.city",
            "tier": "source.tier",
            "signup_date": "source.signup_date",
            "effective_from": F.lit(change_effective_date).cast("date"),
            "effective_to": F.lit(None).cast("date"),
            "is_current": F.lit(True)
        }
    )
    .execute()
)
